# 03 — Sensor-budget analysis

This notebook investigates the central project question: **how much occupancy-detection performance is retained when fewer or cheaper physical sensors are used?**

The packaged experiment evaluates all 15 non-empty combinations of temperature, humidity, light, and CO2 sensors. It compares five model families inside chronological cross-validation and selects one model for each sensor combination.

This notebook does **not** train those models again. It loads the saved experiment tables, explains the results, and creates interactive Plotly visualizations.

If the artifacts are absent, run this command from the project root first:

```powershell
$env:PYTHONPATH = "$PWD\src"
python -m sensorbudget.modeling.sensor_budget
```

## 1. Experiment design

There are four physical sensors, which produce four direct measurements. `HumidityRatio` is different: it is calculated from temperature and relative humidity, so it is included at zero additional sensor cost whenever both required measurements are available.

For each sensor combination, the pipeline:

1. evaluates dummy, logistic-regression, decision-tree, random-forest, and histogram-gradient-boosting models;
2. uses five expanding chronological folds from the training period;
3. selects the model with the highest mean validation F1;
4. refits that model on the full training period; and
5. reports Test 1 and Test 2 separately.

The cost values are transparent **relative assumptions**, not vendor quotations.

In [ ]:
# Import table-processing, display, and interactive Plotly tools.
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

PLOTLY_TEMPLATE = "plotly_white"
FEATURE_LABELS = {
    "light": "Light",
    "temperature": "Temperature",
    "humidity": "Humidity",
    "co2": "CO2",
}


In [ ]:
# Locate the project root whether Jupyter starts there or in notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "models" / "sensor_budget").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

ARTIFACT_DIR = PROJECT_ROOT / "models" / "sensor_budget"
required_files = {
    "scenarios": ARTIFACT_DIR / "sensor_scenarios.csv",
    "folds": ARTIFACT_DIR / "cv_fold_metrics.csv",
    "summary": ARTIFACT_DIR / "cv_summary.csv",
    "selected": ARTIFACT_DIR / "selected_models.csv",
    "heldout": ARTIFACT_DIR / "heldout_metrics.csv",
    "cost_sensitivity": ARTIFACT_DIR / "cost_sensitivity.csv",
}

# Fail with a useful instruction instead of a later, cryptic CSV error.
missing = [str(path) for path in required_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing sensor-budget artifacts. Run "
        "`python -m sensorbudget.modeling.sensor_budget` first.\n"
        + "\n".join(missing)
    )

# Load only result tables; the large row-level prediction file is unnecessary here.
scenarios = pd.read_csv(required_files["scenarios"])
fold_metrics = pd.read_csv(required_files["folds"])
cv_summary = pd.read_csv(required_files["summary"])
selected = pd.read_csv(required_files["selected"])
heldout = pd.read_csv(required_files["heldout"])
cost_sensitivity = pd.read_csv(required_files["cost_sensitivity"])

print(f"Loaded {len(scenarios)} sensor scenarios.")
print(f"Loaded {len(fold_metrics)} fold-level model evaluations.")
print(f"Loaded {len(heldout)} held-out evaluations.")


## 2. Sensor combinations and cost assumptions

The table below makes the experiment inputs auditable. `physical_sensors` determines cost, while `model_features` shows the actual columns supplied to the model.

In [ ]:
# Sort from the least to the most expensive configuration for easier reading.
scenario_view = scenarios[
    [
        "physical_sensors",
        "physical_sensor_count",
        "model_features",
        "derived_features",
        "relative_cost",
    ]
].sort_values(["relative_cost", "physical_sensors"])
display(scenario_view.reset_index(drop=True))


## 3. Which model was selected for each sensor set?

Model selection happens independently for every sensor combination. A simple model may work best for one combination while a nonlinear model works best for another. The bar chart uses mean chronological-validation F1; it does not use the held-out test periods.

In [ ]:
# Create readable sensor labels for plots and hover information.
selected["sensor_label"] = selected["physical_sensors"].apply(
    lambda value: ", ".join(
        FEATURE_LABELS.get(sensor, sensor)
        for sensor in value.split(", ")
    )
)
selected["model_label"] = selected["selected_model"].str.replace(
    "_", " ", regex=False
).str.title()

# Rank configurations by their validation result, not their test result.
ranked = selected.sort_values("cv_f1_mean", ascending=True)
fig = go.Figure()
for model_label, model_rows in ranked.groupby("model_label", sort=False):
    fig.add_trace(
        go.Bar(
            x=model_rows["cv_f1_mean"],
            y=model_rows["sensor_label"],
            orientation="h",
            name=model_label,
            customdata=model_rows[["relative_cost", "physical_sensor_count"]],
            hovertemplate=(
                "%{y}<br>Mean validation F1: %{x:.3f}"
                "<br>Relative cost: %{customdata[0]:.1f}"
                "<br>Sensor count: %{customdata[1]}<extra></extra>"
            ),
        )
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Validation-selected model for every sensor combination",
    xaxis_title="Mean chronological-validation F1",
    yaxis_title="Physical sensors",
    height=650,
    legend_title_text="Selected model",
    barmode="stack",
)
fig.show()


## 4. Performance versus cost and the Pareto frontier

A configuration is **Pareto-efficient** here when no cheaper configuration has a higher mean validation F1. A point can therefore leave the frontier even when it performs well: another configuration may achieve at least as much for less assumed cost.

The frontier is calculated from validation results. Using Test 1 or Test 2 to construct it would turn the held-out data into another model-selection set.

In [ ]:
# Keep only the best validation score available at each exact cost level.
cost_best = (
    selected.groupby("relative_cost", as_index=False)["cv_f1_mean"]
    .max()
    .sort_values("relative_cost")
)

# A point enters the frontier only when it improves on every cheaper point.
previous_best = cost_best["cv_f1_mean"].cummax().shift(fill_value=-np.inf)
frontier = cost_best.loc[cost_best["cv_f1_mean"] > previous_best].copy()

# Match the frontier coordinates back to the named sensor configurations.
selected["is_validation_pareto"] = selected.set_index(
    ["relative_cost", "cv_f1_mean"]
).index.isin(frontier.set_index(["relative_cost", "cv_f1_mean"]).index)
pareto_points = selected.loc[selected["is_validation_pareto"]].sort_values(
    "relative_cost"
)

# Plot every configuration, then connect only the efficient choices.
fig = go.Figure()
for sensor_count, count_rows in selected.groupby("physical_sensor_count"):
    fig.add_trace(
        go.Scatter(
            x=count_rows["relative_cost"],
            y=count_rows["cv_f1_mean"],
            mode="markers",
            name=f"{sensor_count} sensor(s)",
            marker={"size": 8 + 3 * sensor_count},
            customdata=count_rows[["sensor_label", "model_label"]],
            hovertemplate=(
                "%{customdata[0]}<br>Selected model: %{customdata[1]}"
                "<br>Cost: %{x:.1f}<br>Mean validation F1: %{y:.3f}"
                "<extra></extra>"
            ),
        )
    )
fig.add_trace(
    go.Scatter(
        x=pareto_points["relative_cost"],
        y=pareto_points["cv_f1_mean"],
        mode="lines+markers+text",
        text=pareto_points["sensor_label"],
        textposition="top center",
        name="Validation Pareto frontier",
        line={"color": "#222222", "dash": "dash"},
        marker={"color": "#222222", "symbol": "diamond", "size": 11},
        hovertemplate=(
            "%{text}<br>Cost: %{x:.1f}<br>Mean validation F1: %{y:.3f}"
            "<extra></extra>"
        ),
    )
)
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Validation performance versus assumed sensor cost",
    xaxis_title="Illustrative relative sensor cost",
    yaxis_title="Mean chronological-validation F1",
    height=600,
)
fig.show()

display(
    pareto_points[
        ["sensor_label", "relative_cost", "model_label", "cv_f1_mean"]
    ].reset_index(drop=True)
)


## 5. Cost-scenario sensitivity

The previous frontier uses one set of illustrative costs. We now recalculate configuration costs under five explicit scenarios without retraining any model. Validation F1 stays fixed; only the economic assumptions and resulting frontier membership can change.

The exact per-sensor assumptions are shown in the table below. In the cheaper-CO2, expensive-Light, and high-maintenance-CO2 scenarios, only the named sensor changes; the other three retain their current costs. The equal-cost scenario sets every physical sensor to 1.0.

Use the scenario selection buttons below to switch cost assumptions. They behave as a mutually exclusive radio group: one scenario is active at a time. Every point is one sensor configuration at its recalculated cost. The connected green diamonds show the Pareto frontier for the selected scenario. Validation F1 does not move because no model is retrained; only the assumed cost changes.

In [ ]:
# Apply the same readable sensor-name mapping used by the earlier charts.
cost_sensitivity["sensor_label"] = cost_sensitivity[
    "physical_sensors"
].apply(
    lambda value: ", ".join(
        FEATURE_LABELS.get(sensor, sensor)
        for sensor in value.split(", ")
    )
)

# Summarize how often each configuration is efficient across scenarios.
frontier_frequency = (
    cost_sensitivity.groupby("sensor_label", as_index=False)
    .agg(
        frontier_scenarios=("is_pareto", "sum"),
        cv_f1_mean=("cv_f1_mean", "first"),
    )
    .sort_values(["frontier_scenarios", "cv_f1_mean"])
)
scenario_order = cost_sensitivity[
    ["scenario", "scenario_label"]
].drop_duplicates()

# Recover the four individual sensor costs from the single-sensor scenarios.
single_sensor_costs = cost_sensitivity.loc[
    ~cost_sensitivity["physical_sensors"].str.contains(",")
].copy()
scenario_cost_table = (
    single_sensor_costs.pivot(
        index="scenario_label",
        columns="physical_sensors",
        values="scenario_cost",
    )
    .reindex(scenario_order["scenario_label"])
    .rename(columns=FEATURE_LABELS)
    [["Temperature", "Humidity", "Light", "CO2"]]
)
scenario_cost_table.index.name = "Cost scenario"
display(scenario_cost_table)

# Add two traces per scenario: all configurations and its connected frontier.
fig = go.Figure()
for scenario_index, scenario_row in enumerate(
    scenario_order.itertuples(index=False)
):
    scenario_data = cost_sensitivity.loc[
        cost_sensitivity["scenario"] == scenario_row.scenario
    ].sort_values("scenario_cost")
    scenario_frontier = scenario_data.loc[
        scenario_data["is_pareto"]
    ].sort_values("scenario_cost")
    is_visible = scenario_index == 0

    fig.add_trace(
        go.Scatter(
            x=scenario_data["scenario_cost"],
            y=scenario_data["cv_f1_mean"],
            mode="markers",
            name="All configurations",
            visible=is_visible,
            marker={"size": 10, "color": "#9E9E9E", "opacity": 0.75},
            customdata=scenario_data[
                ["sensor_label", "selected_model", "is_pareto"]
            ],
            hovertemplate=(
                "Sensors: %{customdata[0]}"
                "<br>Selected model: %{customdata[1]}"
                "<br>Scenario cost: %{x:.1f}"
                "<br>Mean validation F1: %{y:.3f}"
                "<br>Pareto-efficient: %{customdata[2]}<extra></extra>"
            ),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=scenario_frontier["scenario_cost"],
            y=scenario_frontier["cv_f1_mean"],
            mode="lines+markers",
            name="Pareto frontier",
            visible=is_visible,
            line={"color": "#2A9D8F", "width": 3},
            marker={"size": 12, "symbol": "diamond", "color": "#2A9D8F"},
            customdata=scenario_frontier[["sensor_label", "selected_model"]],
            hovertemplate=(
                "Sensors: %{customdata[0]}"
                "<br>Selected model: %{customdata[1]}"
                "<br>Scenario cost: %{x:.1f}"
                "<br>Mean validation F1: %{y:.3f}<extra></extra>"
            ),
        )
    )

# Each radio-style button reveals exactly the two traces for one scenario.
scenario_buttons = []
for scenario_index, scenario_row in enumerate(
    scenario_order.itertuples(index=False)
):
    scenario_max_cost = float(
        cost_sensitivity.loc[
            cost_sensitivity["scenario"] == scenario_row.scenario,
            "scenario_cost",
        ].max()
    )
    integer_x_max = int(np.ceil(scenario_max_cost))
    visibility = [False] * len(fig.data)
    visibility[2 * scenario_index] = True
    visibility[2 * scenario_index + 1] = True
    scenario_buttons.append(
        {
            "label": {
                "Current assumptions": "Current",
                "Equal sensor costs": "All = 1",
                "Cheaper CO2": "CO2 = 1.5",
                "High Light cost (extreme case)": "Light = 5 (extreme)",
                "High-maintenance CO2": "CO2 = 8",
            }.get(scenario_row.scenario_label, scenario_row.scenario_label),
            "method": "update",
            "args": [
                {"visible": visibility},
                {
                    "title": (
                        "Validation performance versus cost — "
                        + scenario_row.scenario_label
                    ),
                    "xaxis.range": [0, integer_x_max],
                    "xaxis.dtick": 1,
                },
            ],
        }
    )

initial_scenario_label = scenario_order.iloc[0]["scenario_label"]
initial_scenario_name = scenario_order.iloc[0]["scenario"]
initial_x_max = int(
    np.ceil(
        cost_sensitivity.loc[
            cost_sensitivity["scenario"] == initial_scenario_name,
            "scenario_cost",
        ].max()
    )
)
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Validation performance versus cost — " + initial_scenario_label,
    xaxis_title="Illustrative relative sensor cost",
    yaxis_title="Mean chronological-validation F1",
    height=610,
    margin={"t": 90, "r": 30, "b": 70, "l": 80},
    plot_bgcolor="#FAFBFC",
    paper_bgcolor="white",
    legend_title_text="Configuration status",
    updatemenus=[
        {
            "buttons": scenario_buttons,
            "type": "buttons",
            "direction": "right",
            "active": 0,
            "showactive": True,
            "font": {"size": 11},
            "pad": {"r": 2, "t": 2},
            "x": 0.01,
            "xanchor": "left",
            "y": 0.99,
            "yanchor": "top",
            "bgcolor": "rgba(255, 255, 255, 0.92)",
            "bordercolor": "rgba(90, 105, 120, 0.25)",
            "borderwidth": 1,
        }
    ],
)
fig.update_xaxes(
    range=[0, initial_x_max],
    dtick=1,
    showgrid=True,
    gridcolor="rgba(120, 135, 150, 0.16)",
    gridwidth=1,
    zeroline=False,
    showline=True,
    linecolor="rgba(90, 105, 120, 0.35)",
)
fig.update_yaxes(
    range=[0, 1.01],
    dtick=0.2,
    showgrid=True,
    gridcolor="rgba(120, 135, 150, 0.20)",
    gridwidth=1,
    zeroline=False,
    showline=True,
    linecolor="rgba(90, 105, 120, 0.35)",
)
fig.show()

# Report only configurations that enter at least one scenario frontier.
frontier_frequency["frontier_share"] = (
    frontier_frequency["frontier_scenarios"]
    / cost_sensitivity["scenario"].nunique()
)
display(
    frontier_frequency.loc[frontier_frequency["frontier_scenarios"] > 0]
    .sort_values(["frontier_share", "cv_f1_mean"], ascending=False)
)


## 6. Fold-to-fold variability

A mean can hide unstable behavior. The next chart shows the five individual validation-fold F1 scores for the model selected for each sensor combination. Diamonds mark the mean across folds. The supplied training period contains an all-unoccupied weekend fold, where F1 is zero because there are no positive cases to recover.

In [ ]:
# Retain fold rows belonging to the validation-selected model in each scenario.
selected_lookup = selected[["feature_set", "selected_model", "sensor_label"]]
selected_folds = fold_metrics.merge(
    selected_lookup,
    left_on=["feature_set", "model"],
    right_on=["feature_set", "selected_model"],
    how="inner",
)

# Order the boxes by mean validation F1 to match the previous comparison.
sensor_order = selected.sort_values("cv_f1_mean")["sensor_label"].tolist()
fig = go.Figure()

# Give each chronological fold one consistent color across sensor sets.
for fold, fold_rows in selected_folds.groupby("fold"):
    fig.add_trace(
        go.Scatter(
            x=fold_rows["f1"],
            y=fold_rows["sensor_label"],
            mode="markers",
            name=f"Fold {int(fold)}",
            marker={"size": 9, "opacity": 0.85},
            customdata=fold_rows[["validation_occupied_rate", "model"]],
            hovertemplate=(
                "Sensors: %{y}<br>Fold: " + str(int(fold))
                + "<br>Selected model: %{customdata[1]}"
                "<br>F1: %{x:.3f}"
                "<br>Occupied rate: %{customdata[0]:.1%}<extra></extra>"
            ),
        )
    )

# Add one larger diamond per sensor set to make the cross-fold mean visible.
fold_means = (
    selected_folds.groupby("sensor_label", as_index=False)["f1"].mean()
)
fig.add_trace(
    go.Scatter(
        x=fold_means["f1"],
        y=fold_means["sensor_label"],
        mode="markers",
        name="Mean",
        marker={
            "size": 12,
            "symbol": "diamond",
            "color": "#222222",
            "line": {"color": "white", "width": 1},
        },
        hovertemplate=(
            "Sensors: %{y}<br>Mean validation F1: %{x:.3f}"
            "<extra></extra>"
        ),
    )
)
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Chronological variability of each selected configuration",
    xaxis_title="Validation-fold F1",
    yaxis_title="Physical sensors",
    height=650,
    legend_title_text="Validation result",
)
fig.update_yaxes(
    categoryorder="array",
    categoryarray=sensor_order,
)
fig.show()


## 7. Held-out stability

Test 1 and Test 2 occur after the training period. They show whether each validation-selected model remains stable in later conditions. They should be interpreted as confirmation data, not used to redesign the experiment repeatedly.

Each row below contains one dot for Test 1 and one for Test 2. The joining line shows how much F1 changes between periods: a short line means stable performance, while a long line indicates sensitivity to the evaluation period.

In [ ]:
# Reshape the two held-out rows per configuration into one comparison row.
heldout_f1 = heldout.pivot(
    index=["feature_set", "relative_cost", "physical_sensor_count"],
    columns="split",
    values="f1",
).reset_index()
heldout_f1 = heldout_f1.merge(
    selected[["feature_set", "sensor_label", "cv_f1_mean"]],
    on="feature_set",
    how="left",
)

# Build one neutral connector between the two test results on every row.
line_x = []
line_y = []
for row in heldout_f1.itertuples(index=False):
    line_x.extend([row.test_1, row.test_2, None])
    line_y.extend([row.sensor_label, row.sensor_label, None])

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=line_x,
        y=line_y,
        mode="lines",
        line={"color": "#B8B8B8", "width": 3},
        hoverinfo="skip",
        showlegend=False,
    )
)

# Add separate marker traces so the two periods have clear colors and hovers.
for split, color in [("test_1", "#1F77B4"), ("test_2", "#D62728")]:
    display_name = split.replace("_", " ").title()
    fig.add_trace(
        go.Scatter(
            x=heldout_f1[split],
            y=heldout_f1["sensor_label"],
            mode="markers",
            name=display_name,
            marker={"size": 10, "color": color},
            customdata=heldout_f1[
                ["cv_f1_mean", "relative_cost"]
            ],
            hovertemplate=(
                "Sensors: %{y}<br>" + display_name + " F1: %{x:.3f}"
                "<br>Mean validation F1: %{customdata[0]:.3f}"
                "<br>Relative cost: %{customdata[1]:.1f}<extra></extra>"
            ),
        )
    )

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Change in F1 across the two held-out periods",
    xaxis_title="Held-out F1",
    yaxis_title="Physical sensors",
    height=650,
    legend_title_text="Held-out period",
)
fig.update_xaxes(range=[0, 1.02])
fig.update_yaxes(
    categoryorder="array",
    categoryarray=sensor_order,
)
fig.show()


## 8. What changes when Light is added?

The EDA showed that occupied-while-dark observations are almost absent. To make the Light comparison concrete, each row below starts with one fixed set of non-Light sensors and compares its validation F1 before and after adding Light.

A large rightward movement means that adding Light substantially improves validation F1 for that particular base sensor set. This is an observational ablation result, not proof that Light would remain reliable under different room policies or failure conditions.

In [ ]:
# Represent each configuration as a set so matching ignores sensor order.
selected["sensor_key"] = selected["physical_sensors"].apply(
    lambda value: frozenset(value.split(", "))
)
scenario_lookup = {
    row.sensor_key: row for row in selected.itertuples(index=False)
}

# Pair every non-Light configuration with the same sensors plus Light.
light_pairs = []
for base_key, without_light in scenario_lookup.items():
    if "light" in base_key:
        continue
    with_light = scenario_lookup[frozenset(base_key | {"light"})]
    light_pairs.append(
        {
            "base_sensors": without_light.sensor_label,
            "without_light_f1": without_light.cv_f1_mean,
            "with_light_f1": with_light.cv_f1_mean,
            "f1_change": with_light.cv_f1_mean - without_light.cv_f1_mean,
        }
    )
light_pairs = pd.DataFrame(light_pairs).sort_values("f1_change")
# Pre-format the signed change so older Plotly renderers show three decimals.
light_pairs["f1_change_label"] = light_pairs["f1_change"].map(
    lambda value: f"{value:+.3f}"
)

# Draw neutral connectors, then overlay clearly labeled before/after markers.
line_x = []
line_y = []
for row in light_pairs.itertuples(index=False):
    line_x.extend([row.without_light_f1, row.with_light_f1, None])
    line_y.extend([row.base_sensors, row.base_sensors, None])

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=line_x,
        y=line_y,
        mode="lines",
        line={"color": "#B8B8B8", "width": 3},
        hoverinfo="skip",
        showlegend=False,
    )
)
for column, name, color in [
    ("without_light_f1", "Without Light", "#4C78A8"),
    ("with_light_f1", "After adding Light", "#F2A900"),
]:
    fig.add_trace(
        go.Scatter(
            x=light_pairs[column],
            y=light_pairs["base_sensors"],
            mode="markers",
            name=name,
            marker={"size": 11, "color": color},
            customdata=light_pairs[["f1_change_label"]],
            hovertemplate=(
                "Base sensors: %{y}<br>" + name + ": %{x:.3f}"
                "<br>Change after adding Light: %{customdata[0]}"
                "<extra></extra>"
            ),
        )
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Change in validation F1 after adding Light",
    xaxis_title="Mean chronological-validation F1",
    yaxis_title="Base sensors before adding Light",
    height=480,
    legend_title_text="Configuration",
)
fig.update_xaxes(range=[0, 1.02])
fig.update_yaxes(
    categoryorder="array",
    categoryarray=light_pairs["base_sensors"].tolist(),
)
fig.show()


## 9. Conclusions and limitations

- **The highest mean validation F1 is 0.780 for Temperature + Light + CO2.** Light alone reaches 0.755 at much lower assumed cost, while all four physical sensors reach 0.767. These are validation estimates rather than proof of a permanent ranking.
- **Three configurations form the validation Pareto frontier under the current cost assumptions:** Light; Humidity + Light; and Temperature + Light + CO2. The Humidity + Light improvement over Light alone is only about 0.001 F1, so technical Pareto efficiency does not imply a practically meaningful gain.
- **Adding Light improves mean validation F1 in all seven matched comparisons.** The increases range from about 0.059 to 0.543. Because the best model is selected again after Light is added, this measures the complete sensor-plus-model pipeline change; it is not an isolated causal effect of Light.
- **The two held-out periods support the importance of Light in this room, but not universal generalization.** Light-based configurations remain strong in both periods, whereas several no-Light configurations vary substantially. Both periods still come from the same room and collection process.
- **The rankings are uncertain.** Fold-to-fold variation is large, partly because one weekend fold contains no occupied observations. Small differences between validation means should not be treated as decisive.
- **The cost values are scenarios, not prices.** Changing their relative ordering can change the frontier, and an explicit value for additional F1 is needed before judging whether a performance gain justifies extra cost.
- **The core frontier is stable across the five tested cost scenarios.** Light; Humidity + Light; and Temperature + Light + CO2 remain efficient in every scenario. Temperature-only and CO2-only join only in the deliberately extreme high-Light-cost case. This is evidence of scenario stability, not proof against all possible prices or non-additive costs.
- **Phase 4 makes no sensor recommendation.** The frontier configurations are inputs to the robustness phase, not deployment finalists. They must be tested against lights left on while unoccupied, occupied darkness, missing readings, noise, drift, and full sensor loss.
- **The held-out periods should now remain untouched.** Further feature, model, threshold, cost-sensitivity, and robustness decisions should use training-period validation rather than repeatedly optimizing against these test results.

In [ ]:
# Print all validation Pareto points behind the written conclusions.
summary = pareto_points[
    ["sensor_label", "relative_cost", "cv_f1_mean"]
].rename(
    columns={
        "sensor_label": "sensors",
        "cv_f1_mean": "mean_validation_f1",
    }
)
summary["frontier_role"] = "Intermediate frontier point"
summary.loc[summary["relative_cost"].idxmin(), "frontier_role"] = (
    "Lowest-cost frontier point"
)
summary.loc[summary["mean_validation_f1"].idxmax(), "frontier_role"] = (
    "Highest-validation frontier point"
)
summary = summary[
    ["frontier_role", "sensors", "relative_cost", "mean_validation_f1"]
]
display(summary.round({"relative_cost": 1, "mean_validation_f1": 3}))
